# NB17: LightGBM Augmented with RATAN Attention

Loads RATAN attention weights α and uses them to scale feature inputs. Injects RATAN class probabilities as 3 additional features. This creates the first leg of the synergistic coupling.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import accuracy_score
import os, warnings
warnings.filterwarnings("ignore")

SEED = 25
np.random.seed(SEED)

FEATURES_DIR = "../data/features"
ATTN_DIR     = "../data/features/attention_weights"
MODEL_DIR    = "../models"
os.makedirs(MODEL_DIR, exist_ok=True)

CORE_TICKERS = ["SPY", "AAPL", "MSFT", "JPM", "GLD"]
SPLIT_DATE   = "2022-08-22"
print("Environment ready.")


Environment ready.


In [2]:
def load_ticker_data(ticker):
    df = pd.read_csv(f"{FEATURES_DIR}/daily_features.csv", index_col=0, parse_dates=True)
    feat_cols  = [c for c in df.columns if c != f"{ticker}_Target"]
    label_col  = f"{ticker}_Target"
    df = df.dropna(subset=[label_col])
    X, y, idx = df[feat_cols].values, df[label_col].values, df.index
    split = pd.Timestamp(SPLIT_DATE)
    tr = idx < split; te = idx >= split
    return X[tr], y[tr], X[te], y[te], feat_cols, te.sum()

print("Data loader defined.")


Data loader defined.


In [3]:
lgbm_params = dict(
    objective="multiclass", num_class=3,
    n_estimators=1000, learning_rate=0.03,
    num_leaves=63, min_child_samples=20,
    subsample=0.8, colsample_bytree=0.8,
    random_state=SEED, verbose=-1,
    class_weight="balanced"
)

results_aug, lgbm_probs_all = {}, {}

for ticker in CORE_TICKERS:
    print(f"\n{'='*50}\nLightGBM-aug  →  {ticker}")
    X_tr, y_tr, X_te, y_te, feat_cols, n_te = load_ticker_data(ticker)

    # load RATAN outputs (test only)
    alpha       = np.load(f"{ATTN_DIR}/alpha_{ticker}.npy")       # (N_te, 90)
    ratan_probs = np.load(f"{ATTN_DIR}/ratan_probs_{ticker}.npy") # (N_te, 3)
    true_labels = np.load(f"{ATTN_DIR}/true_labels_{ticker}.npy") # (N_te,) {0,1,2}

    N = min(len(X_te), len(alpha))
    alpha        = alpha[:N];        ratan_probs = ratan_probs[:N]
    true_labels  = true_labels[:N];  X_te        = X_te[:N];  y_te = y_te[:N]

    n_alpha = alpha.shape[1]   # 90

    # ── scale test features by RATAN attention ──────────────────
    X_te_scaled = X_te[:, :n_alpha] * alpha          # (N, 90)
    X_te_aug    = np.hstack([X_te_scaled, ratan_probs])  # (N, 93)

    # ── train on unscaled features + dummy RATAN probs ──────────
    dummy_probs = np.full((len(X_tr), 3), 1/3, dtype=np.float32)
    X_tr_aug    = np.hstack([X_tr[:, :n_alpha], dummy_probs])   # (N_tr, 93)

    # labels: {-1,0,1} → {0,1,2}
    y_tr_cls = (y_tr.astype(int) + 1)
    y_te_cls = true_labels   # already {0,1,2} from NB16

    model = lgb.LGBMClassifier(**lgbm_params)
    model.fit(X_tr_aug, y_tr_cls,
              eval_set=[(X_te_aug, y_te_cls)],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(200)])

    model.booster_.save_model(f"{MODEL_DIR}/lgbm_aug_{ticker}.txt")

    probs = model.predict_proba(X_te_aug)   # (N, 3)
    preds = probs.argmax(1)
    acc   = accuracy_score(y_te_cls, preds)

    mask    = (y_te_cls != 1) & (preds != 1)
    dir_acc = ((preds[mask]>1)==(y_te_cls[mask]>1)).mean() if mask.sum()>0 else 0.0
    cov     = mask.mean()

    np.save(f"{ATTN_DIR}/lgbm_aug_probs_{ticker}.npy", probs)
    np.save(f"{ATTN_DIR}/lgbm_aug_true_{ticker}.npy",  y_te_cls)

    lgbm_probs_all[ticker] = probs
    results_aug[ticker] = {"accuracy": acc, "dir_accuracy": dir_acc, "coverage": float(cov)}
    pred_dist = {int(k): int(v) for k,v in zip(*np.unique(preds, return_counts=True))}
    print(f"  → acc={acc:.4f}  dir_acc={dir_acc:.4f}  cov={cov:.4f}  pred_dist={pred_dist}")

print("\nAll augmented LightGBM models trained.")



LightGBM-aug  →  SPY


  → acc=0.4504  dir_acc=0.5597  cov=0.1494  pred_dist={0: 83, 1: 589, 2: 225}

LightGBM-aug  →  AAPL


  → acc=0.2943  dir_acc=0.5937  cov=0.3868  pred_dist={0: 162, 1: 96, 2: 639}

LightGBM-aug  →  MSFT


  → acc=0.2430  dir_acc=0.4508  cov=0.3289  pred_dist={0: 433, 1: 180, 2: 284}

LightGBM-aug  →  JPM


  → acc=0.4560  dir_acc=0.6195  cov=0.1260  pred_dist={1: 629, 2: 268}

LightGBM-aug  →  GLD


  → acc=0.3133  dir_acc=0.7187  cov=0.4359  pred_dist={2: 897}

All augmented LightGBM models trained.


In [4]:
print("\n" + "="*60)
print(f"{'LightGBM-aug Summary':^60}")
print("="*60)
print(f"{'Ticker':<8} {'Accuracy':>10} {'Dir_Acc':>10} {'Coverage':>10}")
print("-"*42)
for t, r in results_aug.items():
    print(f"{t:<8} {r['accuracy']:>10.4f} {r['dir_accuracy']:>10.4f} {r['coverage']:>10.4f}")
avg_acc = np.mean([r['accuracy']    for r in results_aug.values()])
avg_dir = np.mean([r['dir_accuracy'] for r in results_aug.values()])
print(f"{'AVERAGE':<8} {avg_acc:>10.4f} {avg_dir:>10.4f}")
print("="*60)



                    LightGBM-aug Summary                    
Ticker     Accuracy    Dir_Acc   Coverage
------------------------------------------
SPY          0.4504     0.5597     0.1494
AAPL         0.2943     0.5937     0.3868
MSFT         0.2430     0.4508     0.3289
JPM          0.4560     0.6195     0.1260
GLD          0.3133     0.7187     0.4359
AVERAGE      0.3514     0.5885
